# Multivariate polynomials implementations

**Contributed by**: Benoît Legat

The SumOfSquares package is built on top of the [MultivariatePolynomials](https://github.com/JuliaAlgebra/MultivariatePolynomials.jl)
abstract interface. [DynamicPolynomials](https://github.com/JuliaAlgebra/DynamicPolynomials.jl/)
is an implementation of this abstract interface so it can be used with
SumOfSquares. Moreover, any other implementation can be used as well. To
illustrate, we solve of [Blekherman2012; Examples 3.38](@cite) with
[TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl),
another implementation of [MultivariatePolynomials](https://github.com/JuliaAlgebra/MultivariatePolynomials.jl).

In [1]:
import TypedPolynomials
TypedPolynomials.@polyvar x y
using SumOfSquares
import CSDP
model = SOSModel(CSDP.Optimizer)
con_ref = @constraint(model, 2x^4 + 5y^4 - x^2*y^2 >= -2(x^3*y + x + 1))
optimize!(model)
solution_summary(model)

Iter: 18 Ap: 9.59e-01 Pobj: -3.5512000e-03 Ad: 9.59e-01 Dobj: -3.5512523e-03 
Success: SDP solved
Primal objective value: -3.5512000e-03 
Dual objective value: -3.5512523e-03 
Relative primal infeasibility: 1.38e-13 
Relative dual infeasibility: 5.36e-10 
Real Relative Gap: -5.20e-08 
XZ Relative Gap: 1.22e-09 
DIMACS error measures: 1.49e-13 0.00e+00 1.44e-09 0.00e+00 -5.20e-08 1.22e-09
CSDP 6.2.0
This is a pure primal feasibility problem.
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 7.39e-01 Pobj:  0.0000000e+00 Ad: 1.00e+00 Dobj:  9.2771541e+01 
Iter:  2 Ap: 1.00e+00 Pobj:  0.0000000e+00 Ad: 1.00e+00 Dobj:  5.3405053e+01 


solution_summary(; result = 1, verbose = false)
├ solver_name          : CSDP
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : Problem solved to optimality.
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 0.00000e+00
│ └ dual_objective_value : 0.00000e+00
└ Work counters
  └ solve_time (sec)   : 6.83069e-04

We see that the problem is feasible. The Sum-of-Squares decomposition can be
obtained as follows:

In [2]:
sos_decomposition(con_ref)

(-0.6797543969221541 - 0.09754507709792246*y - 0.4175950563713448*x + 2.1931518901396174*y^2 - 0.1269606244678339*x*y - 0.8025867547845221*x^2)^2 + (0.9043573827480671 - 0.6455003192597278*y + 0.2742159536999832*x - 0.10770174782358159*y^2 - 1.2293138068800717*x*y - 0.9300160939847*x^2)^2 + (-0.5739168240154698 - 1.38535050221548*y - 0.6219655378569912*x - 0.2040924100740057*y^2 - 0.15182215880489783*x*y + 0.4443832632766482*x^2)^2 + (0.5703021360913615 - 0.5184093413621602*y + 0.3436728787624473*x + 0.2555950605470665*y^2 + 0.7618441294650047*x*y - 0.02090567376761348*x^2)^2 + (0.041400837217767876 + 0.04911040272637946*y - 0.49990658891857337*x - 0.24792145939081137*y^2 + 0.2972445689917418*x*y - 0.5054184235227418*x^2)^2 + (-0.2524531788141328 - 0.06336647012459033*y + 0.253942572224051*x - 0.10018744153048117*y^2 + 0.059601795985895156*x*y - 0.19381241255811912*x^2)^2

Why is there several implementations ?
Depending in the use-case, one implementation may be more appropriate than
another one. [TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl)
is faster than [DynamicPolynomials](https://github.com/JuliaAlgebra/DynamicPolynomials.jl/)
but it requires new compilation whenever the list of variables changes.
This means that [TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl)
is not appropriate when the number of variables is dynamic or too large.
However, for a small number of variables, it can be faster.
When solving Sum-of-Squares programs, the time is mostly taken by the Semidefinite programming solver.
The time taken by SumOfSquares/JuMP/MathOptInterface are usually negligible
or it time is taken by manipulation of JuMP or MathOptInterface functions
therefore using TypedPolynomials over DynamicPolynomials may not make much difference in most cases.

One case for which using TypedPolynomials might be adequate is when
using domain defined by equalities (possibly also with inequalities).
Computing a Gröbner basis before passing such a domain to SumOfSquares
can take a non-negligible amount of time for large systems of equalities.

To illustrate this, consider the computation of Gröbner basis for the
following system from [CLO05, p. 17].
The time taken by TypedPolynomials is below:

[CLO05] Cox, A. David & Little, John & O'Shea, Donal
*Using Algebraic Geometry*.
Graduate Texts in Mathematics, **2005**.
https://doi.org/10.1007/b138611

In [3]:
using BenchmarkTools
@btime let
    TypedPolynomials.@polyvar x y
    S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
    SemialgebraicSets.compute_gröbner_basis!(S.I)
end

  188.114 μs (7906 allocations: 296.12 KiB)


true

The time taken by DynamicPolynomials is as follows:

In [4]:
import DynamicPolynomials
@btime let
    DynamicPolynomials.@polyvar x y
    S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
    SemialgebraicSets.compute_gröbner_basis!(S.I)
end

  364.119 μs (11632 allocations: 420.13 KiB)


true

We see that TypedPolynomials is faster.
The time is still negligible for this small system but for larger systems, choosing TypedPolynomials may be helpful.
To reduce the polynomial modulo the full ideal, we compute the Gröbner
basis before passing the domain to the SOS constraint.

In [5]:
TypedPolynomials.@polyvar x y
S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
SemialgebraicSets.compute_gröbner_basis!(S.I)
poly = -6x - 4y^3 + 2x*y^2 + 6x^3 - 3y^4 + 13x^2 * y^2
model = Model(CSDP.Optimizer)
con_ref = @constraint(model, poly in SOSCone(), domain = S)
optimize!(model)
solution_summary(model)

Success: SDP solved
Primal objective value: 0.0000000e+00 
Dual objective value: 0.0000000e+00 
Relative primal infeasibility: 2.23e-16 
Relative dual infeasibility: 5.00e-11 
Real Relative Gap: 0.00e+00 
XZ Relative Gap: 3.04e-10 
DIMACS error measures: 2.78e-16 0.00e+00 5.00e-11 0.00e+00 0.00e+00 3.04e-10
CSDP 6.2.0
This is a pure primal feasibility problem.
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 7.64e-01 Pobj:  0.0000000e+00 Ad: 8.07e-01 Dobj:  2.6104524e+00 
Iter:  2 Ap: 7.71e-01 Pobj:  0.0000000e+00 Ad: 7.40e-01 Dobj: -5.9092654e-02 
Iter:  3 Ap: 8.08e-01 Pobj:  0.0000000e+00 Ad: 7.65e-01 Dobj:  5.5549062e-01 
Iter:  4 Ap: 7.56e-01 Pobj:  0.0000000e+00 Ad: 7.79e-01 Dobj:  4.8506833e-01 
Iter:  5 Ap: 6.95e-01 Pobj:  0.0000000e+00 Ad: 6.77e-01 Dobj:  4.3029685e-01 
Iter:  6 Ap: 6.35e-01 Pobj:  0.0000000e+00 Ad: 6.83e-01 Dobj:  2.2137644e-01 
Iter:  7 Ap: 6.21e-01 Pobj:  0.0000000e+00 Ad: 6.43e-01 Dobj:  1.2986387e-01 
Iter:  8 Ap: 

solution_summary(; result = 1, verbose = false)
├ solver_name          : CSDP
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : Problem solved to optimality.
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 0.00000e+00
│ └ dual_objective_value : 0.00000e+00
└ Work counters
  └ solve_time (sec)   : 3.45993e-03

We obtain the following decomposition:

In [6]:
dec = sos_decomposition(con_ref, 1e-6)

(-6.840997569029016e-5 + 0.0006529039515905483*y - 0.00014136210320089623*x + 7.0449656932645865*y^2 - 28.229868930513717*x*y + 25.36384406429942*x^2)^2 + (-1.4193258052395678e-5 + 0.0004273963680194217*y + 0.0008137816963619533*x + 22.47951556126985*y^2 - 13.864997154663644*x*y - 21.67551839144384*x^2)^2 + (-3.8436107538691964e-5 - 0.0007286402052196581*y + 0.0005523227570970247*x + 0.12018521486482847*y^2 + 0.09016325073150745*x*y + 0.0669692094705058*x^2)^2 + (-6.125926173421018e-7 - 0.03996328380700844*y + 0.05340269294283654*x - 0.0002606926912447095*y^2 - 0.00019563475110839876*x*y - 0.00014400542147636526*x^2)^2

We can verify that it is correct as follows:

In [7]:
rem(dec - poly, S.I)

6.3590829805287003104278601415579041944425853216671384871006011962890625e-9 - 2.004589421486302743381201646862373498227436987060802773787425114558293216085126e-7y - 1.645212658073747801469065822087801419771634615384615384615384615384615358042563e-7x - 1.3041658014021686114755453622660041901326621882617473602294921875e-5y² - 1.95303640058753052153629747866858679117285646498203277587890625e-5xy - 7.300456975863297797552618761951492842854349873960018157958984375e-6x² - 4.629880686242889472244617839654286702473958333333333333333333333333335636311615e-8y³ - 4.982772446500878989657697578271230061848958333333333333333333333333332181844193e-8xy² - 4.16282396187721799650331699904626248098793439567089080810546875e-8x²y + 1.238363613363337488128588749812199519230769230769230769230769230769228643404632e-8y⁴

Note that the difference between `dec` and `poly` is larger
than between the full gram matrix because `dec` is obtained by dropping
the lowest eigenvalues with the threshold `1e-6`; see `sos_decomposition`.

In [8]:
rem(gram_matrix(con_ref) - poly, S.I)

7.413492174688583951055860185065660150627309121773578226566314697265625e-9 + 1.593240429062550285715278763998412842193502934279445654306656275040347546044853e-8y - 2.567524517779964718591565122971167931189903846153846153846153846153165889492434e-9x + 7.41358433995441268304915638509555719792842864990234375e-9y² - 3.89098475661597831276594661176204681396484375e-15xy + 7.413443564759092385685335102607496082782745361328125e-9x² - 3.555674273532834680130084355672200520833333333333333333333333333563631161469185e-13y³ - 2.596441580256699429204066594441731770833333333333333333333333333218184419265407e-13xy² + 4.7982451345518484231433831155300140380859375e-15x²y + 1.244138030205459262316043560321514423076923076923076923076923076922532711593844e-8y⁴

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*